In [1]:
import pandas as pd
ratings = pd.read_csv(
    "./ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId", "movieId", "rating", "timestamp"]
)
ratings = ratings[["userId", "movieId", "timestamp", "rating"]]
ratings.head()

,userId,movieId,timestamp,rating
0,1,1193,978300760,5
1,1,661,978302109,3
2,1,914,978301968,3
3,1,3408,978300275,4
4,1,2355,978824291,5


In [2]:
user_ids = ratings["userId"].unique()
movie_ids = ratings["movieId"].unique()

user2idx = {u:i for i, u in enumerate(user_ids)}
movie2idx = {m:i for i, m in enumerate(movie_ids)}

ratings["user"] = ratings["userId"].map(user2idx)
ratings["movie"] = ratings["movieId"].map(movie2idx)

ratings["rating"] = ratings["rating"].astype("float32")
ratings["rating"] = (ratings["rating"] - ratings["rating"].mean()) / ratings["rating"].std()

In [3]:
ratings

,userId,movieId,timestamp,rating,user,movie
0,1,1193,978300760,1.269746,0,0
1,1,661,978302109,-0.520601,0,1
2,1,914,978301968,-0.520601,0,2
3,1,3408,978300275,0.374572,0,3
4,1,2355,978824291,1.269746,0,4
...,...,...,...,...,...,...
1000204,6040,1091,956716541,-2.310948,6039,772
1000205,6040,1094,956704887,1.269746,6039,1106
1000206,6040,562,956704746,1.269746,6039,365
1000207,6040,1096,956715648,0.374572,6039,152


In [4]:
num_users = ratings["user"].nunique()
num_movies = ratings["movie"].nunique()

print(num_users, num_movies)

6040 3706


In [5]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(ratings, test_size = 0.2, random_state = 42)

In [6]:
import torch
import torch.nn as nn
train_user = torch.tensor(train["user"].values)
train_movie = torch.tensor(train["movie"].values)
train_rating = torch.tensor(train["rating"].values, dtype=torch.float32)

In [ ]:
class Recommender(nn.Module):

    def __init__(self, num_users, num_movies, emb_size=50):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_size)
        self.movie_emb = nn.Embedding(num_movies, emb_size)

        self.user_bias = nn.Embedding(num_users, 1)
        self.movie_bias = nn.Embedding(num_movies, 1)

        # ADD THIS
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.movie_emb.weight, std=0.01)
    def forward(self, user, movie):

        u = self.user_emb(user)
        m = self.movie_emb(movie)

        dot = (u * m).sum(1)

        bias = self.user_bias(user).squeeze() + self.movie_bias(movie).squeeze()

        return dot + bias
        #return dot + bias + self.global_bias

In [12]:
model = Recommender(num_users, num_movies)

In [13]:
loss_fn = nn.MSELoss()
initial_lambda = 1e-4
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=initial_lambda
)

for epoch in range(999):

    current_lambda = initial_lambda * (0.95 ** epoch)
    optimizer.param_groups[0]['weight_decay'] = current_lambda

    pred = model(train_user, train_movie)

    loss = loss_fn(pred, train_rating)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch:03d} | Loss {loss.item():.4f}")

Epoch 000 | Loss 2.9920
Epoch 001 | Loss 2.9884
Epoch 002 | Loss 2.9849
Epoch 003 | Loss 2.9814
Epoch 004 | Loss 2.9778
Epoch 005 | Loss 2.9742
Epoch 006 | Loss 2.9705
Epoch 007 | Loss 2.9668
Epoch 008 | Loss 2.9630
Epoch 009 | Loss 2.9590
Epoch 010 | Loss 2.9550
Epoch 011 | Loss 2.9508
Epoch 012 | Loss 2.9465
Epoch 013 | Loss 2.9420
Epoch 014 | Loss 2.9373
Epoch 015 | Loss 2.9325
Epoch 016 | Loss 2.9274
Epoch 017 | Loss 2.9222
Epoch 018 | Loss 2.9168
Epoch 019 | Loss 2.9111
Epoch 020 | Loss 2.9053
Epoch 021 | Loss 2.8992
Epoch 022 | Loss 2.8928
Epoch 023 | Loss 2.8862
Epoch 024 | Loss 2.8794
Epoch 025 | Loss 2.8723
Epoch 026 | Loss 2.8649
Epoch 027 | Loss 2.8573
Epoch 028 | Loss 2.8494
Epoch 029 | Loss 2.8412
Epoch 030 | Loss 2.8327
Epoch 031 | Loss 2.8239
Epoch 032 | Loss 2.8149
Epoch 033 | Loss 2.8055
Epoch 034 | Loss 2.7959
Epoch 035 | Loss 2.7859
Epoch 036 | Loss 2.7756
Epoch 037 | Loss 2.7651
Epoch 038 | Loss 2.7542
Epoch 039 | Loss 2.7430
Epoch 040 | Loss 2.7315
Epoch 041 | Loss

In [14]:
test_user = torch.tensor(test["user"].values)
test_movie = torch.tensor(test["movie"].values)
test_rating = torch.tensor(test["rating"].values, dtype=torch.float32)

In [15]:
import torch.nn.functional as F
import math

model.eval()  # important for evaluation

with torch.no_grad():
    pred = model(test_user, test_movie)
    mse = F.mse_loss(pred, test_rating)
    rmse = math.sqrt(mse.item())

print("Test RMSE:", rmse)

Test RMSE: 0.8560230775846247


Evaluvation

In [33]:
import torch

# Select user 1
user_id = 34

# Get user embedding and bias
user_vec = model.user_emb.weight[user_id]           # shape: [emb_size]
user_b = model.user_bias.weight[user_id].squeeze()  # shape: []

# Get all movie embeddings and biases
movie_vecs = model.movie_emb.weight                 # shape: [num_movies, emb_size]
movie_bs = model.movie_bias.weight.squeeze()       # shape: [num_movies]

In [34]:
# Compute dot product between user and all movies
scores = torch.matmul(movie_vecs, user_vec)  # shape: [num_movies]

# Add biases
scores = scores + user_b + movie_bs 

In [35]:
# Get indices of movies sorted by score descending
ranked_movie_indices = torch.argsort(scores, descending=True)

# Top 10 movies
top10 = ranked_movie_indices[:10]

print("Top 10 recommended movie indices for user 1:", top10.numpy())
print("Corresponding scores:", scores[top10].detach().numpy())

Top 10 recommended movie indices for user 1: [3102 2984 3694 2000 3223 1754  603 1517 2914  611]
Corresponding scores: [1.9295152 1.8277501 1.7595137 1.7083853 1.6813734 1.6122911 1.6109521
 1.6082335 1.6076403 1.5770631]


In [36]:
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)

In [37]:
# Map movieId to index
movie2idx = {m:i for i, m in enumerate(movies["movieId"].unique())}
idx2movie = {i:m for m,i in movie2idx.items()}

# Map index → title
idx2title = {i: movies[movies["movieId"]==mid]["title"].values[0] for i, mid in idx2movie.items()}

In [38]:
# Convert PyTorch tensors to int
top10_int = [int(i) for i in top10]

top10_titles = [idx2title[i] for i in top10_int]

for rank, (title, score) in enumerate(zip(top10_titles, scores[top10].detach().numpy()), start=1):
    print(f"{rank}. {title}  →  score: {score:.3f}")

1. Room at the Top (1959)  →  score: 1.930
2. Messenger: The Story of Joan of Arc, The (1999)  →  score: 1.828
3. F/X (1986)  →  score: 1.760
4. Trip to Bountiful, The (1985)  →  score: 1.708
5. Big Combo, The (1955)  →  score: 1.681
6. Proposition, The (1998)  →  score: 1.612
7. Century (1993)  →  score: 1.611
8. Speed 2: Cruise Control (1997)  →  score: 1.608
9. Ipcress File, The (1965)  →  score: 1.608
10. Bread and Chocolate (Pane e cioccolata) (1973)  →  score: 1.577


In [39]:
# Select test movies for user 1
liked_movies_indices = test[(test["user"] == user_id) & (test["rating"] > 0)]["movie"].values

In [40]:
# Convert to list of ints
liked_movies_int = [int(i) for i in liked_movies_indices]

# Map to titles
liked_movie_titles = [idx2title[i] for i in liked_movies_int]

print("Movies user 1 liked in test set:")
for title in liked_movie_titles:
    print("-", title)

Movies user 1 liked in test set:
- GoldenEye (1995)
- Shadow, The (1994)
- Living in Oblivion (1995)
- From the Journals of Jean Seberg (1995)
- Bullets Over Broadway (1994)
- How to Make an American Quilt (1995)
- Mutters Courage (1995)
- Year of the Horse (1997)
- NeverEnding Story III, The (1994)
- Stag (1997)
- I Love You, I Love You Not (1996)
- Amistad (1997)
- Batman & Robin (1997)
- Once Upon a Time... When We Were Colored (1995)
- Germinal (1993)
- Congo (1995)
- My Favorite Season (1993)
- Wild Bill (1995)
- Mortal Kombat: Annihilation (1997)
- Species (1995)
- Miami Rhapsody (1995)
- FairyTale: A True Story (1997)


In [42]:
recalls = []
for user_id in test["user"].unique():
    liked_movies = test[(test["user"]==user_id) & (test["rating"]>0)]["movie"].values
    if len(liked_movies)==0:
        continue

    all_movies = torch.arange(num_movies)
    user_tensor = torch.tensor([user_id]*num_movies)
    with torch.no_grad():
        scores = model(user_tensor, all_movies)

    seen_movies = train[train["user"]==user_id]["movie"].values
    scores[seen_movies] = -1e9

    top10 = torch.topk(scores, 10).indices.numpy()
    hits = len(set(top10) & set(liked_movies))
    recalls.append(hits / len(liked_movies))

average_recall = sum(recalls)/len(recalls)
print("Average Recall@10:", average_recall)

Average Recall@10: 0.00506278874930469


In [41]:
import torch

K = 10
recalls = []

# For each user
for user_id in ratings["user"].unique():

    # Get all movies rated by this user
    user_movies = ratings[ratings["user"]==user_id].sort_values("timestamp")

    # Leave the last one for test
    test_movie = user_movies.iloc[-1]["movie"]
    train_movies = user_movies.iloc[:-1]["movie"].values

    # Prepare all movies for scoring
    all_movies = torch.arange(num_movies)
    user_tensor = torch.tensor([user_id]*num_movies)

    with torch.no_grad():
        scores = model(user_tensor, all_movies)

    # Remove movies seen in train
    scores[train_movies] = -1e9

    # Top-K recommendations
    topk = torch.topk(scores, K).indices.numpy()

    # Check if the left-out test movie is in top-K
    hit = 1 if test_movie in topk else 0
    recalls.append(hit)

# Average Recall@K
avg_recall_at_10 = sum(recalls)/len(recalls)
print("Average Recall@10 (Leave-One-Out):", avg_recall_at_10)

/tmp/ipykernel_43803/323980312.py:24: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  scores[train_movies] = -1e9


Average Recall@10 (Leave-One-Out): 0.001986754966887417
